# A contracting cube

**Joakim Sundnes**

Date: **June 20, 2025**


## Model outline
This notebook introduces a slight extension of the simple unit cube model introduced previously. The model will still be a simple unit cube, fixed at one end ($x=0$) and loaded with a pressure load (stretch) at the other end ($x=1.0$). The following two extensions will be introduced:
* Replace the St Venant-Kirchhoff model with a model from Guccione et al. (1995). 
* Add a time-varying active stress to the model.

### The material model by Guccione et al.

Soft biological tissues typically follow an exponential stress-strain relation. This relation was originally described by Fung, and has been implemented in a wide range of models for isotropic and anisotropic tissues. One of the most widely used material models for passive cardiac tissue is the model of Guccione et al. from 1995. Several versions of the model have been used in the literature. We apply a transversely isotropic and nearly incompressible version, with strain energy given by:
$$
\begin{align*}
Q &= b_f E_{11}^2 + b_t (E_{22}^2 + E_{33}^2 + E_{23}^2 + E_{32}^2) + b_{fs}(E_{12}^2 + E_{21}^2 + E_{13}^2 + E_{31}^2),\\
\Psi &= \frac{C}{2}(e^Q-1)+ \kappa(J\ln(J)-J+1)  .
\end{align*}
$$
Here $E_{ij}$ are the components of the Green-Lagrange strain tensor, defined relative to the local fiber orientation. Furthermore, $J$ is the determinant of the deformation gradient $F$ ($J=1$ for an incompressible material), and $C,b_f,b_t,b_{fs}, \kappa$ are material parameters. 

### Setting up the FEniCSx solver
The bulk of the solver code will be identical to the first version of the unit cube. First, the usual imports, defining the mesh, the relevant function space and functions, and finally the Neumann and Dirichlet boundary conditions:

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

from dolfinx import fem, geometry, mesh, io, default_scalar_type
import dolfinx.fem.petsc
from ufl import (
    TestFunction,
    Measure,
    FacetNormal,
    variable,
    Identity,
    grad,
    diff,
    dot,
    inner,
    tr,
    det,
    inv,
    dx,
    as_vector,
)
from mpi4py import MPI

from guccionematerial import GuccioneMaterial
from plotting import setup_gif_visualizer, update_gif_frame

In [ ]:
# Setup the mesh and the function space for the solutions
domain = mesh.create_unit_cube(MPI.COMM_WORLD, 4, 4, 4)
V = fem.functionspace(domain, ("CG", 1, (domain.geometry.dim,)))

# Define functions
v = TestFunction(V)  # Test function
u = fem.Function(V, name="u")  # Displacement

# Mark boundary subdomains
fdim = domain.topology.dim - 1


# Define marker functions for the left and right boundaries
def left(x):
    return np.isclose(x[0], 0)


def right(x):
    return np.isclose(x[0], 1.0)


# Locate the degrees of freedom and facets on the left and right boundaries
dofs_l = fem.locate_dofs_geometrical(V, left)
facets_l = mesh.locate_entities_boundary(domain, fdim, left)

dofs_r = fem.locate_dofs_geometrical(V, right)
facets_r = mesh.locate_entities_boundary(domain, fdim, right)

# Generate the meshtags, tagging the left and right boundaries with different markers
marker_l = 1
marker_r = 2

entities = np.hstack([facets_l, facets_r])
values = np.hstack([np.full_like(facets_l, marker_l), np.full_like(facets_r, marker_r)])
boundary_markers = mesh.meshtags(
    domain,
    fdim,
    entities,
    values,
)

# Redefine boundary measure
ds = Measure("ds", domain, subdomain_data=boundary_markers)

# Define Dirichlet boundary condition on left boundary
bc = fem.dirichletbc(np.zeros(3, dtype=default_scalar_type), dofs_l, V)
bcs = [bc]

In [ ]:
track_point = [1.0, 0.5, 0.5]


def evaluate_at_point(mesh, u, point):
    pt_array = np.array([point], dtype=mesh.geometry.x.dtype)
    ownership = geometry.determine_point_ownership(mesh, pt_array, padding=1e-6)
    cells = ownership.dest_cells
    cell_idx = cells[0]
    value = u.eval(pt_array, np.array([cell_idx], dtype=np.int32))
    return value

Next, we turn to defining the mechanics problem. The following code cell is identical to the exercise from yesterday, and goes through the following steps:
* Define the relevant kinematics
* Define the strain energy function
* Define the weak form, including the boundary conditions defined above
* Solve the problem with a for loop, gradually increasing the load
* Store the solution for plotting in ParaView, and plot the displacement in a single point

In [ ]:
# Kinematics
d = len(u)
I = variable(Identity(d))  # Identity tensor
F = variable(I + grad(u))  # Deformation gradient
C = variable(F.T * F)  # the right Cauchy-Green tensor
E = variable(0.5 * (C - I))  # the Green-Lagrange strain tensor

#######################################################################
#  Exercise: Replace the material model with the Guccione material.

# Material parameters (Lamé parameters)
mu = 4.0
lmbda = 20.0

# The strain energy for the St-Venant Kirchhoff model:
psi = lmbda / 2 * (tr(E) ** 2) + mu * tr(E * E)

S = diff(psi, E)  # second Piola-Kirchhoff stress
P = F * S  # First Piola-Kirchhoff stress
########################################################################

p_right = fem.Constant(domain, 0.0)  # the pressure load (zero for now)

# Definition of the weak form:
N = FacetNormal(domain)
traction = p_right * det(F) * dot(inv(F).T, N)
# Residual: Internal forces - External forces
R = inner(grad(v), P) * dx - inner(v, traction) * ds(2)


petsc_options = {
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
    "snes_monitor": None,
}
problem = fem.petsc.NonlinearProblem(
    R,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="nonlinear_basic",
)

# Prepare output file
outfile = io.VTXWriter(domain.comm, "output/active_cube_u.bp", [u])
outfile.write(0.0)
# Prepare GIF
plotter, grid, magnitude, us_expr, actor = setup_gif_visualizer(
    domain, u, filename="output/active_cube.gif"
)

# Finally, we solve the problem for different loads, and plot the load vs displacement.

# Step-wise loading (for plotting and convergence)
load_steps = 5
target_load = 10.0
loads = np.linspace(0, target_load, load_steps)
disps = np.zeros(load_steps)  # array to store displacement for all steps

for step in range(load_steps):
    print(f"Solving for load={loads[step]}")

    ###############################################################
    # Exercise: Update the active tension instead of the pressure load
    
    # Update traction value
    p_right.value = loads[step]
    ###############################################################
    # Solve the problem
    problem.solve()

    # Evaluate displacement at point defined above
    disps[step] = evaluate_at_point(domain, u, track_point)[0]  # extract x comp.

    # Write GIF frame
    update_gif_frame(plotter, grid, u, magnitude, us_expr, actor)

    # Write displacement to file
    outfile.write(loads[step])

outfile.close()
plotter.close()

## Step 1: Replace the St Venant-Kirchhoff material model
Modify the code above to use the Guccione material model. You can implement the strain energy function directly yourself, or you can use an existing Python class which can be found here:

* [Guccione model (1995)](./guccionematerial.py)

The class supports both fully incompressible and nearly incompressible models. The key part is the function named `strain_energy`, which defines the strain energy as a function of the deformation gradient $F$. 

An important difference between the St Venant-Kirchhoff material and the Guccione model is that the Guccione model is anisotropic, meaning that the material properties are different in different directions. The material model therefore needs to know about the local tissue microstructure, i.e., the orientation of the fiber-, sheet- and sheet normal directions. For flexibility and generality, our material model class takes these vectors (or vector fields) as input parameters, as can be seen in the class' constructor.  

In our simple unit cube, it is natural to define the fiber direction as parallel with the x-axis, the sheet direction parallel with the y-axis, and the normal direction parallel with the z-axis. Code for defining these vectors can look as follows:

In [ ]:
# Tissue microstructure
f0 = as_vector([1.0, 0.0, 0.0])
s0 = as_vector([0.0, 1.0, 0.0])
n0 = as_vector([0.0, 0.0, 1.0])

Add these code lines and the necessary calls to the Guccione model class in the code above, to replace the St Venant-Kirchhoff material model with the Guccione material. 

## Step 2: Add active contraction

Next, we want to add active contraction to the tissue cube. If we want to simulate a full cardiac cycle, a simple option would be to assign an active stress transient similar to the one output from the Rice et al. model:

In [ ]:
force_amplitude = 1.0
# Ca_diastolic=0.09
start_time = (5,)
tau1 = 20
tau2 = 110
t = np.linspace(0, 1000, 101)

beta = -math.pow(tau1 / tau2, -1 / (1 - tau2 / tau1)) + math.pow(
    tau1 / tau2, -1 / (-1 + tau1 / tau2)
)
force = (
    (force_amplitude)
    * (np.exp((start_time - t) / tau1) - np.exp((start_time - t) / tau2))
    / beta
)

# the following line implements the if test in numpy without a loop
force = force * (t >= start_time) + 0.0 * (t < start_time)

plt.plot(t, force)
plt.xlabel("Time")
plt.ylabel("Force")
plt.show()

For now, we will keep things even simpler, and simply model the first phase of contraction by adding a linearly increasing active stress. For this case we consider an unloaded cube ($p=0$ on the right boundary). If you used the `GuccioneModel` class above to define the passive material properties, the active stress may be set in the function `set_active_stress`. 

A suitable definition of a linearly increasing active stress can be as follows:

In [ ]:
# Step-wise loading (for plotting and convergence)
active_steps = 6
target_active = 5.0
active = np.linspace(0, target_active, active_steps)

Add these lines and a suitable call to the `set_active_stress` function to the code above, to make the cube contract actively.

### Solution
The code below includes the complete solution for the active cube. 

In [ ]:
# Setup the mesh and the function space for the solutions
domain = mesh.create_unit_cube(MPI.COMM_WORLD, 4, 4, 4)
V = fem.functionspace(domain, ("CG", 1, (domain.geometry.dim,)))

# Define functions
v = TestFunction(V)  # Test function
u = fem.Function(V)  # Displacement from previous iteration

# Mark boundary subdomains
fdim = domain.topology.dim - 1


# Define marker functions for the left and right boundaries
def left(x):
    return np.isclose(x[0], 0)


def right(x):
    return np.isclose(x[0], 1.0)


# Locate the degrees of freedom and facets on the left and right boundaries
dofs_l = fem.locate_dofs_geometrical(V, left)
facets_l = mesh.locate_entities_boundary(domain, fdim, left)

dofs_r = fem.locate_dofs_geometrical(V, right)
facets_r = mesh.locate_entities_boundary(domain, fdim, right)

# Generate the meshtags, tagging the left and right boundaries with different markers
marker_l = 1
marker_r = 2

entities = np.hstack([facets_l, facets_r])
values = np.hstack([np.full_like(facets_l, marker_l), np.full_like(facets_r, marker_r)])
boundary_markers = mesh.meshtags(
    domain,
    fdim,
    entities,
    values,
)

# Redefine boundary measure
ds = Measure("ds", domain, subdomain_data=boundary_markers)

# Define Dirichlet boundary condition on left boundary
clamp = fem.Constant(domain, np.zeros(3, dtype=default_scalar_type))
bc = fem.dirichletbc(clamp, dofs_l, V)
bcs = [bc]

# Define a point (1.0, 0.5, 0.5) on the right boundary to track the displacement in
track_point = [1.0, 0.5, 0.5]

# Kinematics
d = len(u)
I = Identity(d)  # Identity tensor
F = I + grad(u)  # Deformation gradient
F = variable(F)
C = F.T * F  # the right Cauchy-Green tensor
E = 0.5 * (C - I)  # the Green-Lagrange strain tensor

# Material parameters (Lamé parameters)
mu = 4.0
lmbda = 20.0

# The strain energy for the St-Venant Kirchhoff model:
# psi = lmbda / 2 * (tr(E)**2) + mu * tr(E * E)
# P = diff(psi, F)

# Tissue microstructure (needed by the GuccioneMaterial class)
f0 = as_vector([1.0, 0.0, 0.0])
s0 = as_vector([0.0, 1.0, 0.0])
n0 = as_vector([0.0, 0.0, 1.0])

"""
Create an instance of the material model class, differentiate the
class' strain_energy function to get the stress. The parameter
Tactive is set to zero for now, to be update later
"""
material = GuccioneMaterial(domain, e1=f0, e2=s0, e3=n0, kappa=1e2, Tactive=0.0)
psi = material.strain_energy(F)
P = diff(psi, F)

# for this example we keep the pressure at zero, but keep it here for flexibility
p_right = fem.Constant(domain, 0.0)

# Definition of the weak form:
N = FacetNormal(domain)
traction = p_right * det(F) * dot(inv(F).T, N)
R = inner(P, grad(v)) * dx - inner(v, traction) * ds(2)

# Set up nonlinear problem
petsc_options = {
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
    "snes_monitor": None,
}
problem = fem.petsc.NonlinearProblem(
    R,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="nonlinear_basic",
)

# Prepare output file
outfile = io.VTXWriter(domain.comm, "output/active_cube_u.bp", [u])
outfile.write(0.0)
# Prepare GIF
plotter, grid, magnitude, us_expr, actor = setup_gif_visualizer(
    domain, u, filename="output/active_cube.gif", clim=[0, 0.2]
)

# Ramp up the active tension from 0 to 5 in 6 steps
active_steps = 6
target_active = 5.0
active = np.linspace(0, target_active, active_steps, dtype=np.float64)
disps = np.zeros(active_steps)


for step in range(active_steps):
    print(f"Solving for active tension={active[step]}")

    # Update active tension value
    material.set_active_stress(active[step])

    # Solve the nonlinear problem
    problem.solve()

    # Evaluate displacement at point defined above
    disps[step] = evaluate_at_point(domain, u, track_point)[0]  # extract x comp.

    # Write GIF frame
    update_gif_frame(plotter, grid, u, magnitude, us_expr, actor)

    # Write displacement to file
    outfile.write(step)

outfile.close()
plotter.close()

plt.figure()
plt.plot(disps, active)
plt.xlabel(r"Displacement $u_x$ of point (1.0, 0.5, 0.5)")
plt.ylabel("Applied pressure load")
plt.show()

In [ ]:
# Display the generated GIF
from IPython.display import Image

Image(filename="output/active_cube.gif", width=500)